# Step-by-step API example

Prepare and analyze a G-quadruplex folding simulation using individual
package functions and methods. Set inputs in the cells below and run them
in order.

Use the **jupyter_env** kernel with `g4_builder` installed; see the
[installation instructions](../README.md#installation-and-environment).
Activate GROMACS before starting the kernel. Preparation and extraction
require GROMACS; simulation additionally requires PLUMED support.

In [1]:
import g4_builder as g4

## 1. Create the working directory

Choose a new `working_dir`. Relative paths use the kernel’s current directory. This cell creates the directory and configures logging; it raises an error if the directory already exists.

In [4]:
# 1. Define output locations and create the working directory.
# RunPaths helps manage the paths to the input / output files throughout the pipline
paths = g4.RunPaths(
    working_dir="example_results/step_by_step_api",
)
paths.working_dir.mkdir(parents=True, exist_ok=False)

# Full GROMACS output goes to the log; the console shows brief messages.
logger = g4.configure_logging(paths.log)

## 2. Build the DNA strand

Provide a sequence written 5′ to 3′. The returned atom DataFrame is saved as the strand PDB.

In [5]:
# 2. Build the DNA strand from an explicitly provided sequence.
strand_builder = g4.DNAStrandBuilder()

strand = strand_builder.build(
    sequence="GGGTAGGGCGGCGGGGCAGGGT"
)

strand_builder.save_pdb(paths.strand)

## 3. Build references

Provide four ordered G-tracts using 1-based residue indices. Tract order and residue order define template correspondence. The template is generated from the flat a/s layers; the full and partial references are then saved.

In [6]:
# 3. Build the reference using explicitly provided G-tract information.
# The template is built from the packaged flat a/s layers.
references = g4.build_references(
    polarity_pattern="ssa",
    helicity="RH",
    strand=strand,
    GT_resids={
        "GT_1": [11, 1, 2],
        "GT_2": [8, 7, 6],
        "GT_3": [19, 20, 21],
        "GT_4": [15, 14, 13]
    }
)

references.save_full_ref(paths.full_ref)
references.save_partial_refs(paths.working_dir)

Saved ref_1: /home/michal/Symulacje/G4_builder/notebooks/example_results/step_by_step_api/ref_1.pdb
Saved ref_2: /home/michal/Symulacje/G4_builder/notebooks/example_results/step_by_step_api/ref_2.pdb
Saved ref_3: /home/michal/Symulacje/G4_builder/notebooks/example_results/step_by_step_api/ref_3.pdb
Saved ref_4: /home/michal/Symulacje/G4_builder/notebooks/example_results/step_by_step_api/ref_4.pdb
Saved ref_5: /home/michal/Symulacje/G4_builder/notebooks/example_results/step_by_step_api/ref_5.pdb
Saved ref_6: /home/michal/Symulacje/G4_builder/notebooks/example_results/step_by_step_api/ref_6.pdb
Saved ref_7: /home/michal/Symulacje/G4_builder/notebooks/example_results/step_by_step_api/ref_7.pdb
Saved ref_8: /home/michal/Symulacje/G4_builder/notebooks/example_results/step_by_step_api/ref_8.pdb
Saved ref_9: /home/michal/Symulacje/G4_builder/notebooks/example_results/step_by_step_api/ref_9.pdb
Saved ref_10: /home/michal/Symulacje/G4_builder/notebooks/example_results/step_by_step_api/ref_10.pd

## 4. Configure GROMACS

Set the GROMACS executable name or path. The runner uses the working directory and logger created above.

In [7]:
# 4. Configure GROMACS execution.
# Activate the required GROMACS environment before starting Python.
runner = g4.GromacsRunner(
    working_dir=paths.working_dir,
    executable="gmx",
    logger=logger
)

gromacs = g4.GromacsPreparation(
    runner=runner
)

## 5. Inspect available force fields

Display the force-field menu for this installation before choosing the force field in the next cell.

In [8]:
# 5. Check available force fields. 
ff_list = gromacs.discover_force_fields(strand=paths.strand)

for item in ff_list.items():
    print(item)

(1, 'AMBER03 protein, nucleic AMBER94 (Duan et al., J. Comp. Chem. 24, 1999-2012, 2003)')
(2, 'AMBER94 force field (Cornell et al., JACS 117, 5179-5197, 1995)')
(3, 'AMBER96 protein, nucleic AMBER94 (Kollman et al., Acc. Chem. Res. 29, 461-469, 1996)')
(4, 'AMBER99 force field BSC1 (Orozco et al. Nature methods, 13(1), pp.55-58.)')
(5, 'AMBER99 force field BSC1 (Orozco et al. Nature methods, 13(1), pp.55-58.) with Joung-Cheathm ION params')
(6, 'AMBER99 protein, nucleic AMBER94 (Wang et al., J. Comp. Chem. 21, 1049-1074, 2000)')
(7, 'AMBER99SB protein, nucleic AMBER94 (Hornak et al., Proteins 65, 712-725, 2006)')
(8, 'AMBER99SB-ILDN protein, nucleic AMBER94 (Lindorff-Larsen et al., Proteins 78, 1950-58, 2010)')
(9, 'AMBER99SB-ILDN protein, nucleic AMBER94, RNA chiOL3 (Lindorff-Larsen et al., Proteins 78, 1950-58, 2010)')
(10, 'AMBERGS force field (Garcia & Sanbonmatsu, PNAS 99, 2782-2787, 2002)')
(11, 'CHARMM all-atom force field')
(12, 'CHARMM27 all-atom force field (CHARM22 plus CMAP

## 6. Generate topology and coordinates

Replace `force_field=5` with the appropriate menu number or a short force-field name. Menu numbers depend on the installation.

In [9]:
# 6. Generate the topology and processed coordinates.
# Numeric force-field selection discovers the menu automatically.
# Menu numbers depend on the installation; a short force-field name also works.
gromacs.pdb2gmx(
    force_field=5,
    strand=paths.strand
)

INFO: pdb2gmx completed successfully


## 7. Check reference atom numbering

After `pdb2gmx`, remap and resave references only if their atom serial numbers changed. Atom matching uses residue number and atom name.

In [10]:
# 7. Check whether pdb2gmx changed reference atom serial numbers.
# Update and resave references only when necessary.
processed_structure = g4.read_pdb(paths.conf)

if references.needs_serial_remap(processed_structure):
    references.remap_serials(processed_structure)
    # save new remapped references
    references.save_full_ref(paths.full_ref)
    references.save_partial_refs(paths.working_dir)

## 8. Add a box and minimize

Create the box, prepare the energy-minimization TPR and run minimization. The minimization MDP is supplied by the package.

In [11]:
# 8. Add the simulation box, prepare minimization input and minimize.
# The minimization MDP comes from the package resources.
gromacs.add_box()
gromacs.prep_enmin_file()
gromacs.run_enmin()

INFO: editconf completed successfully
INFO: grompp completed successfully
INFO: mdrun completed successfully


## 9. Write the PLUMED protocol

Set the four restraint timings in simulation steps. Stage arrays, core spacing, targets and force constants are derived automatically. Generate the protocol using the processed structure’s atom serials.

In [12]:
# 9. Define the restraint schedule and generate the PLUMED protocol.
# Set the initial ramp duration, first core start and block durations.
# GT hold kappa and release timing are derived automatically.
schedule = g4.ScheduleConfig(
    gt_initial_steps=400_000,
    core_start=1_000_000,
    core_steps=3_000_000,
    tetrads_steps=3_000_000,
)

protocol = g4.build_protocol(
    references=references,
    atom_serials=processed_structure["serial"],
    schedule=schedule,
    print_stride=5000,
    print_file="COLVAR",
)
protocol.write(paths.plumed)

## 10. Prepare the simulation input

Create the MDP and TPR for the protocol plus 1,500,000 additional steps. The saving stride matches the PLUMED print stride. Both preparation and frame selection use the default timestep of 0.002 ps.

In [13]:
# 10. Prepare the simulation MDP and TPR.
# Add 1,500,000 steps after the folding protocol finishes.
# Keep saving_stride equal to the PLUMED print_stride above.
gromacs.prep_sim_file(
    n_steps=protocol.final_steps + 1_500_000,
    saving_stride=5000,
    dielectric_const=15,
)

INFO: grompp completed successfully


## 11. Run the simulation

Run this cell for local simulation with a PLUMED-enabled GROMACS build. For remote execution, skip it and follow the instructions below.

In [ ]:
# 11. Run the simulation.
# timeout=None allows the subprocess to run without a time limit.
gromacs.run_sim(
    timeout=None
)

### Running on another machine

Complete steps 1–10, then copy the **entire working directory** to
the remote machine. Keep the TPR, PLUMED protocol and all reference
PDBs together. With the default filenames, activate a PLUMED-enabled
GROMACS installation and run from inside that directory:

```bash
gmx mdrun -deffnm smd -cpi -v -plumed plumed_protocol.dat -c smd.pdb
```

On a cluster, use this command in the site’s job-submission script.
After the run, copy `COLVAR` and `smd.xtc` back
into the local working directory, retaining the TPR used for the
simulation. Skip the local simulation cell and continue with steps
12–13.

Keep this kernel open while the remote job runs. To resume in a
fresh session, retain `protocol.final_steps` and
`protocol.full_ref_label` and recreate the runner and extractor
with those values. This manual workflow does not write `run.json`
with metadata that allows to restore the last stage of the pipeline.

## 12. Select frames

Read COLVAR and select five distinct post-protocol times with the lowest full-core RMSD. Times are in picoseconds. Selection does not require trajectory files; results are returned and stored in `best_frames`.

In [14]:
# 12. Select the lowest-RMSD frames after the protocol ends.
# Read COLVAR using the protocol duration and full-core RMSD label.
# Trajectories and the TPR are supplied in the extraction step.
frame_extractor = g4.TrajectoryFrameExtractor(runner=runner)

best_frames = frame_extractor.find_best_frames(
    rmsd_label=protocol.full_ref_label,
    protocol_steps=protocol.final_steps,
    colvar=paths.colvar,
    n=5
)

print("Selected frames {time_ps: RMSD}:")
for item in best_frames.items():
    print(item)

Selected frames {time_ps: RMSD}:
(24190, 0.06345)
(24030, 0.063953)
(24330, 0.066135)
(25300, 0.069417)
(24170, 0.069846)


## 13. Extract frames

Extract the selected times from smd.xtc, using the simulation TPR for atom ordering. Output frames are ranked by RMSD and their paths are printed.

In [15]:
# 13. Extract the selected frames as frame_1.pdb, frame_2.pdb, ...
# Use the simulation trajectory and its matching TPR.
frame_extractor.extract_frames(
    trajectories=paths.trajectory,
    structure=paths.tpr
)

INFO: trjconv completed successfully
INFO: trjconv completed successfully
INFO: trjconv completed successfully


Saved frame_1: /home/michal/Symulacje/G4_builder/notebooks/example_results/step_by_step_api/frame_1.pdb
Saved frame_2: /home/michal/Symulacje/G4_builder/notebooks/example_results/step_by_step_api/frame_2.pdb
Saved frame_3: /home/michal/Symulacje/G4_builder/notebooks/example_results/step_by_step_api/frame_3.pdb


INFO: trjconv completed successfully
INFO: trjconv completed successfully


Saved frame_4: /home/michal/Symulacje/G4_builder/notebooks/example_results/step_by_step_api/frame_4.pdb
Saved frame_5: /home/michal/Symulacje/G4_builder/notebooks/example_results/step_by_step_api/frame_5.pdb
